# build_vuln_alias
`stg_ghsa_normalized`(`ghsa_id` + `identifiers_raw`) → `silver_vuln_alias` (GHSA ↔ CVE Crosswalk).
행 생성 규칙: GHSA 1건 → GHSA 자기 자신 alias 1행 + `identifiers[].type == "CVE"` 인 것마다 CVE alias 1행.
Alias는 사실 관계(fact)라 한번 생긴 triple(`canonical_key`,`id_type`,`id_value`)은 값이 바뀌지 않는다 → 신규 행만 append, 기존 행은 건드리지 않는다.

In [ ]:
%run 00_silver_common

In [ ]:
norm_df = spark.table(TBL["stg_ghsa_normalized"]).select(
    "ghsa_id", "identifiers_raw", "published_at", "batch_id", "ingested_at"
)

ghsa_self_rows = norm_df.select(
    F.upper(F.trim(F.col("ghsa_id"))).alias("canonical_key"),
    F.lit("GHSA").alias("id_type"),
    F.col("ghsa_id").alias("id_value"),
    F.lit("ghsa").alias("source"),
    F.coalesce(F.col("published_at"), F.lit(INGESTED_AT)).alias("valid_from"),
    "batch_id", "ingested_at",
)

cve_rows = (
    norm_df
    .withColumn("_ident", F.explode_outer(F.col("identifiers_raw")))
    .filter(F.upper(F.col("_ident.type")) == "CVE")
    .select(
        F.upper(F.trim(F.col("ghsa_id"))).alias("canonical_key"),
        F.lit("CVE").alias("id_type"),
        F.upper(F.trim(F.col("_ident.value"))).alias("id_value"),
        F.lit("ghsa").alias("source"),
        F.coalesce(F.col("published_at"), F.lit(INGESTED_AT)).alias("valid_from"),
        "batch_id", "ingested_at",
    )
)

alias_df = ghsa_self_rows.unionByName(cve_rows).filter(F.col("id_value").isNotNull())


In [ ]:
if table_exists(TBL["vuln_alias"]):
    existing_keys = spark.table(TBL["vuln_alias"]).select("canonical_key", "id_type", "id_value")
    new_rows = alias_df.join(existing_keys, on=["canonical_key", "id_type", "id_value"], how="left_anti")
else:
    new_rows = alias_df

new_count = new_rows.count()
if new_count > 0:
    write_append(new_rows, TBL["vuln_alias"])

log_batch_run("build_vuln_alias", "SUCCESS", record_count=new_count)
print(f"[build_vuln_alias] new_rows={new_count} batch_id={batch_id}")
